[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [SQLModel, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlmodel-deep-dive.html)

# Many to Many


## What you will be able to do

Connect two tables where each side has many of the other, through a link table whose primary key is
both foreign keys. Read and write the connection as lists with `link_model`, and know what that
cannot do: a column on the link is invisible to it. Write the link as a model of its own when it has
something to say, and connect a table to itself. Recognize the failures: the same pair added twice,
a list that accepts an append and stores nothing, and the two errors that arrive when SQLAlchemy
cannot tell which foreign key a relationship meant.


## The idea

### The problem

A hero belongs to one team, and a foreign key on the hero says which. A hero goes on many missions,
and a mission takes many heroes, and there is no column that can hold that. Putting `mission_id` on
the hero allows one mission; putting a list of hero ids in a column on the mission is not a thing a
relational database does, and every question about it would be a search inside text.

What holds it is a third table, one row for each pair. That is the whole mechanism, and it is worth
being clear that the table is not a technicality: it is where anything about the pairing belongs. A
hero on a mission has a role, joined on a date, and may have been withdrawn, and none of those
belong to the hero or to the mission. They belong to the pairing, which is to say the row in the
middle.

The two ways of writing it in SQLModel differ on exactly that point. `link_model` gives both sides a
list of the other, and the link table stays out of sight, which is right when it holds nothing but
the two ids. When the link has something of its own, the list cannot set it, and the answer is to
stop hiding the table and use it directly.

### What a link model is

> A **link model** is a table model whose columns are the two foreign keys, both marked
> **`primary_key=True`**, so the pair cannot be repeated. **`Relationship(link_model=...)`** on each
> side gives a list of the other side, reading through that table and writing rows into it.
> An **association object** is the same table used as a model: relationships to it on both sides,
> so a row in the middle is made, read and changed like any other, with whatever columns of its own
> it has. The two ways can be mixed, with the lists marked **`viewonly=True`** for reading.

### Why it works that way

- **The pair is the key.** Both columns being the primary key is what stops a hero being added to
  one mission twice, and it is the database that enforces it.
- **`link_model` writes the link row for you.** `mission.heroes.append(hero)` inserts a row with the
  two ids and nothing else, which is why a third column on the link is left empty.
- **An association object is a row you hold.** `Deployment(hero=..., operation=..., role="lead")` is
  an object with a role in it, and the session writes it like any other.
- **Two foreign keys to one table need help.** With two ways to join, SQLAlchemy refuses to choose,
  and the relationship has to say which columns it means.
- **A list and an association object over the same table conflict.** One of them has to be
  `viewonly=True`, or both will try to write the same rows.

### Where this shows up

Tags, permissions, memberships, enrollments, anything between two lists of things. The
**SQLAlchemy, Deep Dive** guide's Many to Many notebook is the long version, where the same link
table carries a status and a grade. The **Loading and N+1** notebook is next, and a list of lists is
where the number of queries becomes worth counting.

### What this notebook covers

- The table in the middle, and the key that is both columns
- Lists on both sides with `link_model`
- What the link table holds, and the column the lists cannot fill
- The link as a model of its own
- A table joined to itself
- Which shape to write
- Missions staffed and reported, finished
- Four failures, from a pair added twice to a relationship that cannot choose a foreign key

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read
it, and read the output underneath it. Everything from Setup onward is where you start running
things, and the rest of the notebook takes this apart piece by piece.

```python
from sqlmodel import Field, Relationship, Session, SQLModel, create_engine


class HeroMissionLink(SQLModel, table=True):
    hero_id: int | None = Field(default=None, foreign_key="hero.id", primary_key=True)
    mission_id: int | None = Field(default=None, foreign_key="mission.id", primary_key=True)


class Hero(SQLModel, table=True):
    id: int | None = Field(default=None, primary_key=True)
    name: str
    missions: list["Mission"] = Relationship(back_populates="heroes", link_model=HeroMissionLink)


class Mission(SQLModel, table=True):
    id: int | None = Field(default=None, primary_key=True)
    title: str
    heroes: list[Hero] = Relationship(back_populates="missions", link_model=HeroMissionLink)


engine = create_engine("sqlite://")
SQLModel.metadata.create_all(engine)
with Session(engine) as session:
    bridge = Mission(title="Rescue the bridge")
    bridge.heroes.append(Hero(name="Deadpond"))
    bridge.heroes.append(Hero(name="Spider-Boy"))
    session.add(bridge)
    session.commit()

    print("on the mission:", [hero.name for hero in bridge.heroes])
    print("the hero's side:", [mission.title for mission in bridge.heroes[0].missions])
```

```
on the mission: ['Deadpond', 'Spider-Boy']
the hero's side: ['Rescue the bridge']
```

Three classes and two lists. The link model has no id of its own: its primary key is both foreign
keys together, which is what makes a pair unrepeatable. Appending to one list wrote two rows in the
middle table, and reading the other list found them from the other end.


## Setup

Twelve imports, one of them installed first where it is missing, the cast, four helpers, six
classes, the engine, and the database built and loaded.

- `sqlmodel` is the library, and `SQLModel`, `Field`, `Relationship`, `Session`, `create_engine` and
  `select`, from it, are the classes, the lists, the session and the reading. Colab does not have
  SQLModel, so the cell installs 0.0.42 with `pip` where it is missing, and `version` and
  `PackageNotFoundError`, from `importlib.metadata`, `subprocess` and `sys` find out whether it is
- `event`, `insert` and `text`, from `sqlalchemy`, are the pragma on every connection, the rows
  `build` loads without a session, and the reads of the link tables themselves
- `IntegrityError`, from `sqlalchemy.exc`, is what a pair added twice raises, and `CreateTable`,
  from `sqlalchemy.schema`, with `sqlite`, from `sqlalchemy.dialects`, writes the link table's SQL
- `re` takes memory addresses out of a message, `Path` names the database file, and `shutil` removes
  the scratch folder at the start and at the end
- `subprocess` and `sys` also run `run_python`, which runs a file in a Python of its own, for the
  two Common errors that break a mapper for the rest of the process they happen in
- `TEAMS` and `HEROES` are the cast, which `build` loads

Six classes, because this notebook shows both ways of writing the table in the middle and one of
each is not enough. `HeroMissionLink` connects heroes and missions and carries a `role` nothing can
fill; `Deployment` connects heroes and operations and is a model in its own right. The classes are
written once, since a class defined a second time cannot be resolved by name, as the
**Relationships** notebook's last error shows.


In [1]:
import re
import shutil
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

try:
    version("sqlmodel")
except PackageNotFoundError:                                        # Colab has no SQLModel: install the pinned version
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--root-user-action=ignore",
                    "sqlmodel==0.0.42"], check=True)

import sqlmodel
from sqlalchemy import event, insert, text
from sqlalchemy.dialects import sqlite
from sqlalchemy.exc import IntegrityError
from sqlalchemy.schema import CreateTable
from sqlmodel import Field, Relationship, Session, SQLModel, create_engine, select

TEAMS = [                                                           # name, headquarters
    ("Preventers", "Sharp Tower"),
    ("Z-Force", "Sister Margaret's Bar"),
    ("Wakaland Guard", "Grand Palace"),                             # no heroes, for the joins that keep a team anyway
]

HEROES = [                                                          # name, secret name, age, team
    ("Deadpond", "Dive Wilson", None, "Z-Force"),
    ("Spider-Boy", "Pedro Parqueador", 16, "Preventers"),
    ("Rusty-Man", "Tommy Sharp", 48, "Preventers"),
    ("Tarantula", "Natalia Roman-on", 32, "Preventers"),
    ("Black Lion", "Trevor Challa", 35, "Z-Force"),
    ("Dr. Weird", "Steve Weird", 36, "Z-Force"),
    ("Captain North America", "Esteban Rogelios", 93, "Preventers"),
    ("Princess Sure-E", "Sure-E", None, None),                      # on no team
]


def message(error):
    """An error's text, without the memory address or the version link that make no two runs agree."""
    text = re.sub(r"0x[0-9a-f]+", "0x...", str(error))
    return "\n".join(line for line in text.splitlines() if "errors.pydantic.dev" not in line).strip()


def fields(model):
    """A model's values in the order its class declares them, which a loaded object does not keep."""
    return {name: getattr(model, name) for name in type(model).model_fields}


def table_sql(model):
    """The CREATE TABLE a table model describes, written for SQLite with no database anywhere."""
    return str(CreateTable(model.__table__).compile(dialect=sqlite.dialect())).strip()

def run_python(path):
    """Run a file in a Python of its own and print what it printed."""
    done = subprocess.run([sys.executable, path], capture_output=True, text=True)
    print(done.stdout.strip() or done.stderr.strip().splitlines()[-1])


class HeroMissionLink(SQLModel, table=True):
    """The table in the middle, with a key made of both sides."""

    hero_id: int | None = Field(default=None, foreign_key="hero.id", primary_key=True)
    mission_id: int | None = Field(default=None, foreign_key="mission.id", primary_key=True)
    role: str | None = Field(default=None, max_length=30)           # nothing can set this through the lists


class Deployment(SQLModel, table=True):
    """The same shape, written as a model with two attributes of its own."""

    hero_id: int | None = Field(default=None, foreign_key="hero.id", primary_key=True)
    operation_id: int | None = Field(default=None, foreign_key="operation.id", primary_key=True)
    role: str = Field(max_length=30)

    hero: "Hero" = Relationship(back_populates="deployments")
    operation: "Operation" = Relationship(back_populates="deployments")


class Team(SQLModel, table=True):
    id: int | None = Field(default=None, primary_key=True)
    name: str = Field(index=True, max_length=50)
    headquarters: str = Field(max_length=60)

    heroes: list["Hero"] = Relationship(back_populates="team")


class Hero(SQLModel, table=True):
    id: int | None = Field(default=None, primary_key=True)
    name: str = Field(index=True, max_length=50)
    secret_name: str = Field(max_length=60)
    age: int | None = Field(default=None, index=True)
    team_id: int | None = Field(default=None, foreign_key="team.id")

    team: Team | None = Relationship(back_populates="heroes")
    missions: list["Mission"] = Relationship(back_populates="heroes", link_model=HeroMissionLink)
    deployments: list[Deployment] = Relationship(back_populates="hero")


class Mission(SQLModel, table=True):
    id: int | None = Field(default=None, primary_key=True)
    title: str = Field(unique=True, max_length=80)

    heroes: list[Hero] = Relationship(back_populates="missions", link_model=HeroMissionLink)


class Operation(SQLModel, table=True):
    id: int | None = Field(default=None, primary_key=True)
    code_name: str = Field(unique=True, max_length=80)

    deployments: list[Deployment] = Relationship(back_populates="operation")


def hero_engine(path=None, echo=False):
    """The guide's engine: the database in a file, or with no path one in memory, with foreign keys checked."""
    engine = create_engine("sqlite://" if path is None else f"sqlite:///{path}", echo=echo)

    @event.listens_for(engine, "connect")
    def enforce_foreign_keys(connection, record):
        cursor = connection.cursor()
        cursor.execute("PRAGMA foreign_keys=ON")
        cursor.close()

    return engine


def build(engine):
    """Create the tables and load the cast, without a session: every notebook starts from the same rows."""
    SQLModel.metadata.create_all(engine)
    with engine.begin() as connection:
        connection.execute(insert(Team), [{"name": name, "headquarters": where} for name, where in TEAMS])
        teams = {name: number for number, (name, _) in enumerate(TEAMS, start=1)}
        connection.execute(insert(Hero), [{"name": name, "secret_name": secret, "age": age,
                                           "team_id": teams.get(team)}
                                          for name, secret, age, team in HEROES])

shutil.rmtree("scratch", ignore_errors=True)                        # a rerun starts from the same eight heroes
Path("scratch").mkdir()
engine = hero_engine("scratch/heroes.db")
build(engine)

with Session(engine) as session:
    print("sqlmodel", sqlmodel.__version__, "|", len(session.exec(select(Hero)).all()), "heroes |",
          "tables:", sorted(SQLModel.metadata.tables))


sqlmodel 0.0.42 | 8 heroes | tables: ['deployment', 'hero', 'heromissionlink', 'mission', 'operation', 'team']


## Worked examples

### The table in the middle, and the key that is both columns

The link table has no id. Its primary key is the pair:


In [2]:
print(table_sql(HeroMissionLink))
print()
print("primary key:", [column.name for column in HeroMissionLink.__table__.primary_key])


CREATE TABLE heromissionlink (
	hero_id INTEGER NOT NULL, 
	mission_id INTEGER NOT NULL, 
	role VARCHAR(30), 
	PRIMARY KEY (hero_id, mission_id), 
	FOREIGN KEY(hero_id) REFERENCES hero (id), 
	FOREIGN KEY(mission_id) REFERENCES mission (id)
)

primary key: ['hero_id', 'mission_id']


Two foreign keys, both in the primary key, and a `role` that this notebook comes back to. A primary
key of two columns means the database will not hold the same pair twice, which is the first of the
Common errors, and it means there is no id to refer to a pairing by: the pair is its name.

### Lists on both sides with link_model

`Hero.missions` and `Mission.heroes` are lists of the other side, reading through that table:


In [3]:
with Session(engine) as session:
    bridge = Mission(title="Rescue the bridge")
    deadpond = session.get(Hero, 1)
    spider = session.get(Hero, 2)
    bridge.heroes.append(deadpond)
    bridge.heroes.append(spider)
    session.add(bridge)
    session.commit()

    print("on the mission:", sorted(hero.name for hero in bridge.heroes))
    print("Deadpond's     :", [mission.title for mission in deadpond.missions])


on the mission: ['Deadpond', 'Spider-Boy']
Deadpond's     : ['Rescue the bridge']


Two existing heroes and one new mission, connected by appending to a list. Nothing mentioned the
link table, and nothing mentioned an id: the session wrote the mission, took the id the database
gave it, and inserted a row in the middle for each hero.

### What the link table holds, and the column the lists cannot fill

The rows are there, and the `role` is not:


In [4]:
with Session(engine) as session:
    print(session.connection().exec_driver_sql(
        "select hero_id, mission_id, role from heromissionlink").fetchall())


[(1, 1, None), (2, 1, None)]


`append` puts the two ids in and has nowhere to put anything else, so every role is null. There is
no argument to give it: the list is a list of heroes, and the role belongs to the pairing rather
than to either side.

Writing the link row directly is one answer, and it works:


In [5]:
with Session(engine) as session:
    session.add(HeroMissionLink(hero_id=3, mission_id=1, role="lookout"))
    session.commit()
    print(session.connection().exec_driver_sql(
        "select hero_id, mission_id, role from heromissionlink").fetchall())

    bridge = session.get(Mission, 1)
    print("the list sees it:", sorted(hero.name for hero in bridge.heroes))


[(1, 1, None), (2, 1, None), (3, 1, 'lookout')]
the list sees it: ['Deadpond', 'Rusty-Man', 'Spider-Boy']


The row is in, the list reads it, and the role is set. What it is not is comfortable: the ids are
written by hand, so the caller has to have them, and nothing stops the two ways of writing from
being mixed in one program until somebody appends and loses a role.

### The link as a model of its own

`Deployment` is the same shape with the table left in plain sight: relationships to the hero and to
the operation, and a role that is required rather than optional:


In [6]:
with Session(engine) as session:
    lead, lookout = session.get(Hero, 1), session.get(Hero, 5)      # looked up before anything is built
    nightfall = Operation(code_name="Nightfall")
    nightfall.deployments.append(Deployment(hero=lead, role="lead"))
    nightfall.deployments.append(Deployment(hero=lookout, role="lookout"))
    session.add(nightfall)
    session.commit()

    for deployment in session.get(Operation, 1).deployments:
        print(f"  {deployment.hero.name:<12} as {deployment.role}")


  Deadpond     as lead
  Black Lion   as lookout


The role is given where the pairing is made, because the pairing is an object. Both heroes are
looked up before the first `Deployment` is built, for a reason worth keeping: any query sends what
the session is holding to the database first, and a graph that is half built is not ready to go.
Reading is the same in reverse: a hero's `deployments` are rows in the middle, each holding the
operation and the role.

The cost is one more step to get from a hero to an operation, and that is the trade. `link_model`
hides the table and cannot carry anything; an association object shows the table and carries
whatever it needs.


In [7]:
with Session(engine) as session:
    deadpond = session.get(Hero, 1)
    print("through deployments:", [f"{d.operation.code_name} as {d.role}" for d in deadpond.deployments])
    print("through missions   :", [mission.title for mission in deadpond.missions])


through deployments: ['Nightfall as lead']
through missions   : ['Rescue the bridge']


### A table joined to itself

Heroes who work with heroes is the same shape with one table on both sides, and it needs one thing
more: two foreign keys to `hero.id`, and a relationship that says which of them is which end. That
one runs in a Python of its own, since the classes here are already defined:


In [8]:
%%writefile scratch/allies.py
from sqlmodel import Field, Relationship, Session, SQLModel, create_engine, select


class Alliance(SQLModel, table=True):
    hero_id: int | None = Field(default=None, foreign_key="hero.id", primary_key=True)
    ally_id: int | None = Field(default=None, foreign_key="hero.id", primary_key=True)


class Hero(SQLModel, table=True):
    id: int | None = Field(default=None, primary_key=True)
    name: str = Field(max_length=50)

    allies: list["Hero"] = Relationship(
        link_model=Alliance,
        sa_relationship_kwargs={"primaryjoin": "Hero.id == Alliance.hero_id",
                                "secondaryjoin": "Hero.id == Alliance.ally_id"})


engine = create_engine("sqlite://")
SQLModel.metadata.create_all(engine)
with Session(engine) as session:
    deadpond = Hero(name="Deadpond")
    deadpond.allies.append(Hero(name="Rusty-Man"))
    deadpond.allies.append(Hero(name="Black Lion"))
    session.add(deadpond)
    session.commit()

    print("allies of Deadpond:", sorted(hero.name for hero in session.get(Hero, 1).allies))
    print("the link rows     :", session.connection().exec_driver_sql("select * from alliance").fetchall())


Writing scratch/allies.py


In [9]:
run_python("scratch/allies.py")


allies of Deadpond: ['Black Lion', 'Rusty-Man']
the link rows     : [(1, 2), (1, 3)]


`primaryjoin` says which column is this hero and `secondaryjoin` which is the other, written as text
because the class is not finished being defined when the relationship is written. Without them the
relationship has two ways to join and refuses to guess, which is the third of the Common errors.

Note that the list is one-directional: an ally added to Deadpond does not gain Deadpond. Making it
mutual means writing both rows, which is a decision about what an alliance is rather than something
the library settles.

### Which shape to write

| The link holds | Write | What you get |
|---|---|---|
| the two ids and nothing else | `Relationship(link_model=...)` on both sides | a list on each side, the middle table hidden |
| a column of its own | an association object: relationships to the link, from both sides | the pairing as an object, with its columns |
| both, for convenience | the association object, plus lists marked `viewonly=True` | reading either way, writing one way |
| two foreign keys to one table | `primaryjoin` and `secondaryjoin` in `sa_relationship_kwargs` | a table joined to itself |

Start with `link_model`, and move to an association object the moment the pairing gains a column.
Moving later is a migration and a change to every place that appends, which is why a link that looks
likely to gain a column is worth writing as a model from the start.

### Missions staffed and reported, finished

The pieces of this notebook in two functions. `staff` puts a hero on an operation with a role, and
`report` reads a mission and an operation from both sides:


In [10]:
def staff(session, hero_name, code_name, role):
    """Put a hero on an operation, with the role that pairing has."""
    hero = session.exec(select(Hero).where(Hero.name == hero_name)).one()
    operation = session.exec(select(Operation).where(Operation.code_name == code_name)).one()
    if any(deployment.hero_id == hero.id for deployment in operation.deployments):
        return f"{hero_name} is already on {code_name}"
    operation.deployments.append(Deployment(hero=hero, role=role))
    session.commit()
    return f"{hero_name} is on {code_name} as {role}"


def report(session, code_name):
    """Who is on an operation, and what else each of them is doing."""
    operation = session.exec(select(Operation).where(Operation.code_name == code_name)).one()
    return {"operation": operation.code_name,
            "staff": sorted(f"{d.hero.name} ({d.role})" for d in operation.deployments),
            "also on missions": sorted({mission.title for d in operation.deployments
                                        for mission in d.hero.missions})}


with Session(engine) as session:
    print(staff(session, "Rusty-Man", "Nightfall", "driver"))
    print(staff(session, "Deadpond", "Nightfall", "lead"))
    print(report(session, "Nightfall"))


Rusty-Man is on Nightfall as driver
Deadpond is already on Nightfall
{'operation': 'Nightfall', 'staff': ['Black Lion (lookout)', 'Deadpond (lead)', 'Rusty-Man (driver)'], 'also on missions': ['Rescue the bridge']}


The guard is the notebook's first Common error, avoided: a pair that is already there would be
refused by the database, so the function looks before it appends. The report walks three
relationships: the operation's deployments, the hero on each of them, and that hero's missions. As
the **Loading and N+1** notebook is about to show, that is rather more queries than it looks.

### Where each part came from

| In `staff` and `report` | What it relies on | The section that showed it |
|---|---|---|
| `operation.deployments.append(Deployment(hero=hero, role=role))` | the link as a model, with a column of its own | The link as a model of its own |
| `deployment.hero_id` | the link row's own columns, readable before a commit | The table in the middle |
| `deployment.hero.name` | a relationship from the link row to one side | The link as a model of its own |
| `hero.missions` | a list through `link_model`, read from the other end | Lists on both sides with `link_model` |


## Your turn

Six tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than getting
there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlmodel-deep-dive/09-many-to-many-solutions.ipynb).

**1.** Add a mission called Guard the vault with three heroes of your choosing, and print its heroes
and the link rows it wrote.


In [11]:
# your code here


**2.** Print every hero who is on at least one mission, with the number of missions, using the
relationship.


In [12]:
# your code here


**3.** Take one hero off Guard the vault by removing them from the list, and show the link row is
gone.


In [13]:
# your code here


**4.** Put two heroes on a new operation called Daybreak with roles of your choosing, and print the
operation's deployments as `name (role)`.


In [14]:
# your code here


**5.** Print every role each hero has ever had, from the hero's side, as `name: role, role`.


In [15]:
# your code here


**6.** Write `missions_of(session, hero_name)`, returning the titles of a hero's missions and the
code names of their operations in one dictionary.


In [16]:
# your code here


## Common errors

### sqlalchemy.exc.IntegrityError: (sqlite3.IntegrityError) UNIQUE constraint failed: heromissionlink.hero_id, heromissionlink.mission_id


In [17]:
with Session(engine) as session:
    bridge = session.get(Mission, 1)
    bridge.heroes.append(session.get(Hero, 2))                      # Spider-Boy is already on it
    session.commit()


IntegrityError: (sqlite3.IntegrityError) UNIQUE constraint failed: heromissionlink.hero_id, heromissionlink.mission_id
[SQL: INSERT INTO heromissionlink (hero_id, mission_id) VALUES (?, ?)]
[parameters: (2, 1)]
(Background on this error at: https://sqlalche.me/e/20/gkpj)

The primary key of the link table is the pair, so the database refuses the second row. That is the
constraint doing its job: a hero is on a mission or is not, and there is no meaning to being on it
twice.

The session is now waiting for a rollback, as the **Sessions** notebook showed. Looking first is
what the capstone does, and it costs nothing, since the list is loaded anyway:


In [18]:
with Session(engine) as session:
    session.rollback()
    bridge = session.get(Mission, 1)
    spider = session.get(Hero, 2)
    if spider in bridge.heroes:
        print(spider.name, "is already on", bridge.title)
    else:
        bridge.heroes.append(spider)
        session.commit()


Spider-Boy is already on Rescue the bridge


### No error, and a hero who was never deployed: an append that writes nothing


In [19]:
%%writefile scratch/viewonly.py
from sqlmodel import Field, Relationship, Session, SQLModel, create_engine, select


class Deployment(SQLModel, table=True):
    hero_id: int | None = Field(default=None, foreign_key="hero.id", primary_key=True)
    operation_id: int | None = Field(default=None, foreign_key="operation.id", primary_key=True)
    role: str = Field(max_length=30)

    hero: "Hero" = Relationship(back_populates="deployments")
    operation: "Operation" = Relationship(back_populates="deployments")


class Hero(SQLModel, table=True):
    id: int | None = Field(default=None, primary_key=True)
    name: str = Field(max_length=50)
    deployments: list[Deployment] = Relationship(back_populates="hero")


class Operation(SQLModel, table=True):
    id: int | None = Field(default=None, primary_key=True)
    code_name: str = Field(max_length=80)

    deployments: list[Deployment] = Relationship(back_populates="operation")
    heroes: list[Hero] = Relationship(link_model=Deployment,        # for reading, beside the deployments
                                      sa_relationship_kwargs={"viewonly": True})


engine = create_engine("sqlite://")
SQLModel.metadata.create_all(engine)
with Session(engine) as session:
    nightfall = Operation(code_name="Nightfall")
    session.add(nightfall)
    session.commit()

    nightfall.heroes.append(Hero(name="Rusty-Man"))                 # the list is a view
    session.commit()

    print("the operation's heroes:", [hero.name for hero in session.get(Operation, 1).heroes])
    print("rows in the middle    :", session.connection().exec_driver_sql(
        "select hero_id, operation_id, role from deployment").fetchall())
    print("heroes written        :", len(session.exec(select(Hero)).all()))


Writing scratch/viewonly.py


In [20]:
run_python("scratch/viewonly.py")


the operation's heroes: []
rows in the middle    : []
heroes written        : 0


A list marked `viewonly=True` reads the link table and never writes to it. The append raised
nothing, the commit raised nothing, and nothing at all was written: no row in the middle, and not
even the hero, since a list that is a view does not put what is appended to it into the session.

The lists have to be a view whenever an association object writes the same table, or the two would
both be writing those rows. So the rule that comes with it is to write through the deployments and
read through the lists, which is what the previous worked examples do.

### sqlalchemy.exc.AmbiguousForeignKeysError: Could not determine join condition between parent/child tables on relationship Hero.allies - there are multiple foreign key paths linking the tables via secondary table 'alliance'.


In [21]:
%%writefile scratch/ambiguous.py
from sqlalchemy.orm import configure_mappers
from sqlmodel import Field, Relationship, SQLModel


class Alliance(SQLModel, table=True):
    hero_id: int | None = Field(default=None, foreign_key="hero.id", primary_key=True)
    ally_id: int | None = Field(default=None, foreign_key="hero.id", primary_key=True)


class Hero(SQLModel, table=True):
    id: int | None = Field(default=None, primary_key=True)
    name: str = Field(max_length=50)

    allies: list["Hero"] = Relationship(link_model=Alliance)        # which column is which end?


try:
    configure_mappers()
except Exception as error:
    print(type(error).__module__ + "." + type(error).__name__)
    print(str(error).splitlines()[0])


Writing scratch/ambiguous.py


In [22]:
run_python("scratch/ambiguous.py")


sqlalchemy.exc.AmbiguousForeignKeysError
Could not determine join condition between parent/child tables on relationship Hero.allies - there are multiple foreign key paths linking the tables via secondary table 'alliance'.  Specify the 'foreign_keys' argument, providing a list of those columns which should be counted as containing a foreign key reference from the secondary table to each of the parent and child tables.


Both columns of the link point at `hero.id`, so there are two ways to join a hero to the table and
SQLAlchemy will not pick one. `primaryjoin` and `secondaryjoin` are what say which is which, as the
worked example above does.

It is worth being clear about what does not cause this: a link model with extra columns is not
ambiguous, and neither is a link between two different tables. It takes two foreign keys to the same
table.

### sqlalchemy.exc.AmbiguousForeignKeysError: Could not determine join condition between parent/child tables on relationship Team.heroes - there are multiple foreign key paths linking the tables.


In [23]:
%%writefile scratch/two_keys.py
from sqlalchemy.orm import configure_mappers
from sqlmodel import Field, Relationship, SQLModel


class Team(SQLModel, table=True):
    id: int | None = Field(default=None, primary_key=True)
    name: str = Field(max_length=50)

    heroes: list["Hero"] = Relationship(back_populates="team")


class Hero(SQLModel, table=True):
    id: int | None = Field(default=None, primary_key=True)
    name: str = Field(max_length=50)
    team_id: int | None = Field(default=None, foreign_key="team.id")
    backup_team_id: int | None = Field(default=None, foreign_key="team.id")     # a second way in

    team: Team | None = Relationship(back_populates="heroes")


try:
    configure_mappers()
except Exception as error:
    print(type(error).__module__ + "." + type(error).__name__)
    print(str(error).splitlines()[0])


Writing scratch/two_keys.py


In [24]:
run_python("scratch/two_keys.py")


sqlalchemy.exc.AmbiguousForeignKeysError
Could not determine join condition between parent/child tables on relationship Team.heroes - there are multiple foreign key paths linking the tables.  Specify the 'foreign_keys' argument, providing a list of those columns which should be counted as containing a foreign key reference to the parent table.


The same refusal without a link table in sight. A hero has two columns pointing at `team.id`, so
`Team.heroes` has two ways to find its heroes. The message is the one above with the last clause
changed, and the fix is the argument it names:


In [25]:
%%writefile scratch/two_keys_fixed.py
from sqlalchemy.orm import configure_mappers
from sqlmodel import Field, Relationship, SQLModel


class Team(SQLModel, table=True):
    id: int | None = Field(default=None, primary_key=True)
    name: str = Field(max_length=50)

    heroes: list["Hero"] = Relationship(back_populates="team",
                                        sa_relationship_kwargs={"foreign_keys": "[Hero.team_id]"})


class Hero(SQLModel, table=True):
    id: int | None = Field(default=None, primary_key=True)
    name: str = Field(max_length=50)
    team_id: int | None = Field(default=None, foreign_key="team.id")
    backup_team_id: int | None = Field(default=None, foreign_key="team.id")

    team: Team | None = Relationship(back_populates="heroes",
                                     sa_relationship_kwargs={"foreign_keys": "[Hero.team_id]"})


configure_mappers()
print("the mappers configured, and the join is", Team.heroes.property.primaryjoin)


Writing scratch/two_keys_fixed.py


In [26]:
run_python("scratch/two_keys_fixed.py")


the mappers configured, and the join is team.id = hero.team_id


Both sides of the pair name the same column, because they are two views of one join. A second
relationship for `backup_team_id`, with its own `foreign_keys`, is how the other column gets a list
of its own.

Last, the engine lets go of the file, and this cell removes the scratch folder with the database and
the files in it:


In [27]:
engine.dispose()
shutil.rmtree("scratch")

print("scratch still there:", Path("scratch").exists())


scratch still there: False


## Recap

- Many to many needs a table in the middle, and a link model's primary key is both foreign keys, so
  a pair cannot be repeated.
- `Relationship(link_model=...)` on both sides gives a list of the other side and writes the link
  rows for you, with nothing to put in any other column.
- When the link has a column of its own, make it a model with relationships on both sides, and make
  it the thing you write.
- Lists over a table an association object also writes have to be `viewonly=True`, and an append to
  one of those writes nothing at all.
- Two foreign keys to one table leave a relationship with two ways to join: `primaryjoin` and
  `secondaryjoin` for a link table, `foreign_keys` for a plain one.


## What is next

The **Loading and N+1** notebook counts the queries the lists in this notebook send. Ten teams and
two hundred heroes, one loop, two hundred and one statements, and what `selectinload` and
`lazy="selectin"` each do about it.


---

&#8592; **Previous:** [Relationships](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlmodel-deep-dive/08-relationships.ipynb)  &nbsp;·&nbsp;  [SQLModel, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlmodel-deep-dive.html)
